# Power Profiling - CodeCarbon

**Objective:** Estimate per-condition power draw for the active device profile using
[CodeCarbon](https://github.com/mlco2/codecarbon) software-based energy tracking,
when a physical USB power meter (TC66C) is unavailable or impractical.

**Methodology difference from `04_power_profiling.ipynb`:**
- `04_power_profiling.ipynb` uses a **hardware** power meter (JT-TC66C) via BLE/serial
- This notebook uses **CodeCarbon's `EmissionsTracker`**, which estimates energy from
  CPU/GPU hardware counters when available, or falls back to model-based estimates

**Protocol (aligned with hardware measurements):**
- **Warm-up:** 60 s per condition (inference runs but energy is discarded)
- **Measurement window:** 180 s per condition
- **Repetitions:** 5 runs per condition
- **Reporting:** mean +- 95 % CI across runs

**Output:** `results/profiling/table1_power_<result_tag>.csv` - same schema as
the TC66C-based power tables, enabling direct comparison in `03_results_figures.ipynb`.

**Prerequisites:**
- `codecarbon` installed: `pip install codecarbon`
- The target profile is either auto-detected or selected via `DEVICE_PROFILE`
- Models for that profile are already exported under `models/`
- MOT17 data is available under `data/`
- `table1_profiling_<result_tag>.csv` is optional and only needed for the Table I merge

In [ ]:
import time
import tempfile
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from codecarbon import EmissionsTracker

from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQ_SUFFIX
from benchmark.device_profile import load_profile, apply_thread_pinning, iter_model_imgsz

SUPPORTED_BACKENDS = {"cpu", "cuda", "ncnn", "hailo", "tensorrt_hq"}

# -- Device profile ---------------------------------------------------------
PROFILE = load_profile()
if PROFILE.backend not in SUPPORTED_BACKENDS:
    raise RuntimeError(
        "This notebook currently supports CodeCarbon runs for backends "
        f"{sorted(SUPPORTED_BACKENDS)}; active backend is {PROFILE.backend!r}. "
        "QNN is not wired here yet because the repository does not include "
        "a benchmark.qnn_runner implementation."
    )
apply_thread_pinning(PROFILE)

# -- Output paths -----------------------------------------------------------
POWER_DIR = RESULTS_RAW.parent / "power" / PROFILE.result_tag
PROFILING_DIR = RESULTS_RAW.parent / "profiling"
FIGURES_DIR = RESULTS_RAW.parent / "figures"
POWER_DIR.mkdir(parents=True, exist_ok=True)

# -- Measurement parameters -------------------------------------------------
WARMUP_DURATION_S = 60.0
MEASURE_DURATION_S = 180.0
N_REPS = 5
CODECARBON_INTERVAL = 15
SKIP_EXISTING = True

# -- IEEE two-column figure style ------------------------------------------
matplotlib.rcParams.update({
    "font.size": 10, "axes.titlesize": 10, "axes.labelsize": 10,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "lines.linewidth": 1.2, "axes.linewidth": 0.8,
})

print(f"Device      : {PROFILE.device_label}")
print(f"Profile     : {PROFILE.profile_path}")
print(f"Backend     : {PROFILE.backend}")
print(f"Tag         : {PROFILE.result_tag}")
print(f"Power dir   : {POWER_DIR}")
print(f"Method      : CodeCarbon (software energy estimation)")
print(f"Protocol    : {WARMUP_DURATION_S:.0f}s warm-up + {MEASURE_DURATION_S:.0f}s measure x {N_REPS} reps")

## Helpers

In [ ]:
from __future__ import annotations


def _model_stem(model_path: str) -> str:
    """Base model name without extension, resolution suffix, or backend tag.
    yolo26n_640_ncnn_model -> yolo26n
    yolo26n_576_qnn.onnx   -> yolo26n
    yolo26n.pt             -> yolo26n
    """
    stem = model_path.rsplit(".", 1)[0]
    for tag in ("_ncnn_model", "_hq", "_lq", "_qnn"):
        if stem.endswith(tag):
            stem = stem[: -len(tag)]
            break
    parts = stem.rsplit("_", 1)
    return parts[0] if len(parts) == 2 and parts[-1].isdigit() else stem


def _power_csv(model_path: str, imgsz: int, rep: int | None = None) -> Path:
    """CSV path for a single CodeCarbon repetition."""
    stem = _model_stem(model_path)
    if rep is not None:
        return POWER_DIR / f"{stem}_{imgsz}_rep{rep:02d}.csv"
    return POWER_DIR / f"{stem}_{imgsz}.csv"


def _rep_csvs(model_path: str, imgsz: int) -> list[Path]:
    """All existing repetition CSVs for a condition, sorted by rep index."""
    stem = _model_stem(model_path)
    pattern = f"{stem}_{imgsz}_rep*.csv"
    return sorted(POWER_DIR.glob(pattern))


def _condition_done(model_path: str, imgsz: int) -> bool:
    return len(_rep_csvs(model_path, imgsz)) >= N_REPS


def compute_ci95(values: list[float]) -> tuple[float, float]:
    """Mean and 95% CI half-width for a list of values."""
    arr = np.array(values)
    mean = np.mean(arr)
    if len(arr) > 1:
        sem = stats.sem(arr)
        ci = sem * stats.t.ppf((1 + 0.95) / 2.0, len(arr) - 1)
    else:
        ci = 0.0
    return float(mean), float(ci)


def extract_power_from_codecarbon(emissions_csv: Path, duration_s: float) -> dict:
    """Extract mean power (W) from a CodeCarbon emissions CSV.

    CodeCarbon logs cumulative energy_consumed in kWh.
    Power = energy / time.
    """
    df = pd.read_csv(emissions_csv)
    if df.empty:
        return {"mean_W": np.nan, "energy_kWh": np.nan, "duration_s": duration_s}

    row = df.iloc[-1]
    energy_kwh = row.get("energy_consumed", np.nan)
    actual_duration = row.get("duration", duration_s)

    if pd.isna(energy_kwh) or actual_duration <= 0:
        return {"mean_W": np.nan, "energy_kWh": np.nan, "duration_s": actual_duration}

    mean_w = (energy_kwh * 3_600_000) / actual_duration

    return {
        "mean_W": round(mean_w, 4),
        "energy_kWh": energy_kwh,
        "duration_s": actual_duration,
    }


conditions = list(iter_model_imgsz(PROFILE))
print(f"{len(conditions)} conditions x {N_REPS} reps = {len(conditions) * N_REPS} total measurements\n")
for mp, sz in conditions:
    n_done = len(_rep_csvs(mp, sz))
    status = f"DONE ({n_done}/{N_REPS})" if n_done >= N_REPS else f"{n_done}/{N_REPS}"
    print(f"  {_model_stem(mp):12s}  {sz}px  [{status}]")

## Idle Baseline

CodeCarbon measures system-level energy. The idle baseline captures the device's
quiescent power so we can compute delta-W (inference overhead) later.

In [ ]:
# -- Idle baseline: 60 s with no workload ---------------------------------
IDLE_CSV = POWER_DIR / "idle_codecarbon.csv"
IDLE_DURATION_S = 60.0

if IDLE_CSV.exists() and SKIP_EXISTING:
    print(f"Idle baseline loaded from {IDLE_CSV}")
else:
    print(f"Collecting idle baseline ({IDLE_DURATION_S:.0f}s) via CodeCarbon ...")
    tracker = EmissionsTracker(
        output_file=str(IDLE_CSV),
        measure_power_secs=CODECARBON_INTERVAL,
        log_level="warning",
    )
    tracker.start()
    time.sleep(IDLE_DURATION_S)
    tracker.stop()
    print(f"Done - saved to {IDLE_CSV}")

idle_stats = extract_power_from_codecarbon(IDLE_CSV, IDLE_DURATION_S)
IDLE_MEAN_W = idle_stats["mean_W"]
print(f"Idle: mean={IDLE_MEAN_W:.3f} W  energy={idle_stats['energy_kWh']:.6f} kWh  "
      f"duration={idle_stats['duration_s']:.1f} s")

## On-Device Measurement

For each (model, resolution) condition, run the active backend on MOT17-09 (525 frames)
in a continuous loop while CodeCarbon tracks energy consumption.

Each repetition:
1. **Warm-up** (60 s) - inference runs, CodeCarbon tracks, but this energy is discarded
2. **Measurement** (180 s) - a fresh `EmissionsTracker` captures energy for this window only

Runner dispatch is profile-driven:
- `cpu`, `cuda`, `ncnn` -> `benchmark.runner.run_sequence()`
- `hailo` -> `benchmark.hailo_runner.run_sequence_hailo()`
- `tensorrt_hq` -> `benchmark.trt_runner.run_sequence_trt()`

The warm-up uses a separate tracker instance so the measurement tracker focuses on the
steady-state window. Accelerator-specific runners still include any per-phase runtime setup cost.

In [ ]:
# -- Inference + CodeCarbon energy measurement -----------------------------
from benchmark.device_profile import try_load_model, resolve_model_path
from benchmark.runner import run_sequence

if PROFILE.backend == "hailo":
    from benchmark.hailo_runner import run_sequence_hailo
elif PROFILE.backend == "tensorrt_hq":
    from benchmark.trt_runner import run_sequence_trt

POWER_SEQ = "MOT17-09"
POWER_SEQ_DIR = DATA_ROOT / f"{POWER_SEQ}-{SEQ_SUFFIX}"

INFERENCE_SCRATCH = POWER_DIR / "_scratch"
INFERENCE_SCRATCH.mkdir(parents=True, exist_ok=True)

GENERIC_RUNNER_BACKENDS = {"cpu", "cuda", "ncnn"}


def _run_power_workload(model_path: str, imgsz: int, out_csv: Path,
                        max_duration_s: float, model=None) -> None:
    if PROFILE.backend == "hailo":
        run_sequence_hailo(
            resolve_model_path(model_path), POWER_SEQ_DIR,
            imgsz=imgsz, out_csv=out_csv, max_duration_s=max_duration_s,
        )
        return

    if PROFILE.backend == "tensorrt_hq":
        run_sequence_trt(
            resolve_model_path(model_path), POWER_SEQ_DIR,
            imgsz=imgsz, out_csv=out_csv, max_duration_s=max_duration_s,
        )
        return

    if PROFILE.backend in GENERIC_RUNNER_BACKENDS:
        if model is None:
            raise RuntimeError(
                f"Loaded model instance required for backend {PROFILE.backend!r}."
            )
        run_sequence(
            model, POWER_SEQ_DIR, imgsz=imgsz, out_csv=out_csv,
            max_duration_s=max_duration_s,
        )
        return

    raise RuntimeError(f"Unsupported CodeCarbon backend: {PROFILE.backend!r}")


for model_path, imgsz in conditions:
    stem = _model_stem(model_path)

    if SKIP_EXISTING and _condition_done(model_path, imgsz):
        n_done = len(_rep_csvs(model_path, imgsz))
        print(f"  {stem} {imgsz}px: skip (all {n_done} reps exist)")
        continue

    print(f"\n== {stem} @ {imgsz}px {'=' * 34}")

    for rep in range(N_REPS):
        csv_path = _power_csv(model_path, imgsz, rep=rep)

        if SKIP_EXISTING and csv_path.exists():
            print(f"  rep {rep:02d}: skip (exists)")
            continue

        print(f"  rep {rep:02d}/{N_REPS-1}: {WARMUP_DURATION_S:.0f}s warm-up + {MEASURE_DURATION_S:.0f}s measure ...")

        model = None
        if PROFILE.backend in GENERIC_RUNNER_BACKENDS:
            model, err = try_load_model(model_path, PROFILE.torch_device)
            if err:
                print(f"    SKIP - could not load: {err}")
                break

        warmup_csv = INFERENCE_SCRATCH / f"warmup_{stem}_{imgsz}_rep{rep:02d}.csv"
        warmup_emissions = tempfile.NamedTemporaryFile(suffix=".csv", delete=False)
        warmup_emissions.close()
        warmup_tracker = EmissionsTracker(
            output_file=warmup_emissions.name,
            measure_power_secs=CODECARBON_INTERVAL,
            log_level="warning",
        )
        warmup_tracker.start()
        try:
            _run_power_workload(
                model_path, imgsz, warmup_csv, WARMUP_DURATION_S, model=model,
            )
        finally:
            try:
                warmup_tracker.stop()
            except Exception:
                pass
            warmup_csv.unlink(missing_ok=True)
            Path(warmup_emissions.name).unlink(missing_ok=True)

        measure_csv = INFERENCE_SCRATCH / f"measure_{stem}_{imgsz}_rep{rep:02d}.csv"
        tracker = EmissionsTracker(
            output_file=str(csv_path),
            measure_power_secs=CODECARBON_INTERVAL,
            log_level="warning",
        )
        tracker.start()
        try:
            _run_power_workload(
                model_path, imgsz, measure_csv, MEASURE_DURATION_S, model=model,
            )
        finally:
            try:
                tracker.stop()
            except Exception as e:
                print(f"    WARNING: CodeCarbon stop failed: {e}")
            measure_csv.unlink(missing_ok=True)
            if model is not None:
                del model

        if csv_path.exists() and csv_path.stat().st_size > 50:
            power = extract_power_from_codecarbon(csv_path, MEASURE_DURATION_S)
            print(f"    -> {power['mean_W']:.3f} W  ({power['energy_kWh']:.6f} kWh in {power['duration_s']:.1f}s)")
        else:
            print(f"    WARNING: CodeCarbon output missing or empty at {csv_path}")

    n_done = len(_rep_csvs(model_path, imgsz))
    print(f"  {stem} {imgsz}px: {n_done}/{N_REPS} reps complete")

import shutil
shutil.rmtree(INFERENCE_SCRATCH, ignore_errors=True)
print("\nDone - scratch inference output cleaned up.")

## Aggregate Power Statistics

Load all repetition CSVs per condition, extract power from each CodeCarbon output,
then compute mean +- 95% CI across runs - same schema as `04_power_profiling.ipynb`.

In [ ]:
# -- Aggregate CodeCarbon results across repetitions ----------------------
power_rows = []

for model_path, imgsz in conditions:
    stem = _model_stem(model_path)
    rep_files = _rep_csvs(model_path, imgsz)

    if not rep_files:
        print(f"  WARNING: no power data for {stem} {imgsz}px")
        continue

    run_powers = []
    for csv_path in rep_files:
        stats = extract_power_from_codecarbon(csv_path, MEASURE_DURATION_S)
        if not np.isnan(stats["mean_W"]):
            run_powers.append(stats["mean_W"])

    if not run_powers:
        print(f"  WARNING: all reps empty for {stem} {imgsz}px")
        continue

    mean_w, ci95_w = compute_ci95(run_powers)
    std_w = float(np.std(run_powers, ddof=1)) if len(run_powers) > 1 else 0.0
    median_w = float(np.median(run_powers))
    iqr_w = float(np.percentile(run_powers, 75) - np.percentile(run_powers, 25))
    peak_w = float(np.max(run_powers))

    power_rows.append({
        "model": model_path,
        "imgsz": imgsz,
        "mean_W": round(mean_w, 4),
        "ci95_W": round(ci95_w, 4),
        "std_W": round(std_w, 4),
        "median_W": round(median_w, 4),
        "iqr_W": round(iqr_w, 4),
        "peak_W": round(peak_w, 4),
        "delta_W": round(mean_w - IDLE_MEAN_W, 4),
        "n_runs": len(run_powers),
    })

power_df = pd.DataFrame(power_rows)
print(f"Power profile - {PROFILE.device_label}  ({N_REPS} reps, 95% CI, CodeCarbon)\n")
print(power_df[["model", "imgsz", "mean_W", "ci95_W", "median_W", "iqr_W", "delta_W", "n_runs"]]
      .to_string(index=False))

## Merge with Table I

In [ ]:
# -- Merge with existing profiling table ----------------------------------
table1_path = PROFILING_DIR / f"table1_profiling_{PROFILE.result_tag}.csv"

if not table1_path.exists():
    print(f"WARNING: {table1_path.name} not found - run notebook 01 first.")
    print("Saving power-only table instead.")
    out_path = PROFILING_DIR / f"table1_power_{PROFILE.result_tag}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    power_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
else:
    table1 = pd.read_csv(table1_path)

    table1["stem"] = table1["model"].apply(_model_stem)
    power_df["stem"] = power_df["model"].apply(_model_stem)

    agg = (
        table1.groupby(["stem", "imgsz"])
        .agg(fps=("fps", "mean"), mota=("mota", "mean"), idf1=("idf1", "mean"))
        .reset_index()
        .round({"fps": 1, "mota": 3, "idf1": 3})
    )

    merged = agg.merge(power_df, on=["stem", "imgsz"], how="left")
    merged["fps_per_W"] = (merged["fps"] / merged["mean_W"]).round(2)

    out_path = PROFILING_DIR / f"table1_power_{PROFILE.result_tag}.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_path, index=False)

    print(f"Saved: {out_path}\n")
    print(merged[["stem", "imgsz", "fps", "mota", "mean_W", "ci95_W", "delta_W", "fps_per_W", "n_runs"]]
          .to_string(index=False))

## Figures

In [ ]:
# -- Figure: power draw vs model complexity -------------------------------
MODEL_ORDER = ["yolo26n", "yolo26s", "yolo26m", "yolo26l", "yolo26x"]
MODEL_LABELS = {"yolo26n": "N", "yolo26s": "S", "yolo26m": "M",
                "yolo26l": "L", "yolo26x": "X"}
IMGSZ_COLORS = {640: "#1f77b4", 576: "#ff7f0e", 512: "#2ca02c",
                448: "#d62728", 384: "#9467bd", 320: "#8c564b"}

fig, ax = plt.subplots(figsize=(3.5, 2.8))

plot_df = power_df.copy()
plot_df["stem"] = plot_df["model"].apply(_model_stem)

for imgsz, grp in plot_df.groupby("imgsz"):
    grp = grp.set_index("stem").reindex(MODEL_ORDER).dropna(subset=["mean_W"])
    x = range(len(grp))
    ax.errorbar(
        x, grp["mean_W"], yerr=grp["ci95_W"],
        fmt="o-", color=IMGSZ_COLORS.get(imgsz, "grey"),
        label=f"{imgsz}px", markersize=4, capsize=3,
    )

ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
ax.set_xlabel("Model variant")
ax.set_ylabel("Power draw (W)")
ax.set_title(f"Estimated power - {PROFILE.device_label} (CodeCarbon)")
ax.legend(fontsize=7, ncol=2, title="95% CI", title_fontsize=6)
ax.grid(axis="y", linewidth=0.4, alpha=0.5)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
for fmt in ("pdf", "png"):
    path = FIGURES_DIR / f"fig_power_{PROFILE.result_tag}.{fmt}"
    fig.savefig(path, bbox_inches="tight", dpi=300)
    print(f"Saved: {path}")
plt.show()

In [ ]:
# -- Figure: inference efficiency (fps / W) -------------------------------
if "merged" not in dir():
    print("Skipped - Table I merge was not available (run notebook 01 first).")
else:
    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    eff_df = merged.copy()
    eff_df["stem"] = eff_df["model"].apply(_model_stem) if "model" in eff_df.columns else eff_df["stem"]

    for imgsz, grp in eff_df.groupby("imgsz"):
        grp = grp.set_index("stem").reindex(MODEL_ORDER).dropna(subset=["fps_per_W"])
        x = range(len(grp))
        ax.plot(
            x, grp["fps_per_W"], "s--",
            color=IMGSZ_COLORS.get(imgsz, "grey"),
            label=f"{imgsz}px", markersize=4,
        )

    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels([MODEL_LABELS[m] for m in MODEL_ORDER])
    ax.set_xlabel("Model variant")
    ax.set_ylabel("Efficiency (fps / W)")
    ax.set_title(f"Inference efficiency - {PROFILE.device_label} (CodeCarbon)")
    ax.legend(fontsize=7, ncol=2)
    ax.grid(axis="y", linewidth=0.4, alpha=0.5)

    for fmt in ("pdf", "png"):
        path = FIGURES_DIR / f"fig_efficiency_{PROFILE.result_tag}.{fmt}"
        fig.savefig(path, bbox_inches="tight", dpi=300)
        print(f"Saved: {path}")
    plt.show()

## Observations

- **Method:** CodeCarbon software energy estimation (RAPL / powercap / TDP fallback)
- **Caveat:** CodeCarbon estimates CPU package power, not total system power.
  The TC66C hardware meter used for other devices measures USB rail power (CPU + peripherals + board).
  Direct cross-device comparison requires noting this methodological difference.
- Protocol: 60 s warm-up + 180 s measure x 5 reps per condition
- Idle baseline: *(fill after run)*
- Delta-W range: *(fill after run)*
- Mean 95% CI width: *(fill after run)*
- Most efficient condition (fps/W): *(fill after run)*

## Next steps

- Re-run on another target by changing `DEVICE_PROFILE` (or relying on auto-detection) and executing from the top
- Results merge into `03_results_figures.ipynb` via `table1_power_<result_tag>.csv`
- Document the TC66C vs CodeCarbon methodology difference in the paper